In [25]:
import cadquery as cq
from jupyter_cadquery import show


def create_triangle_outline(part, outer_thickness=1):
    
    bbox = part.val().BoundingBox()
    
    def outline_line(inp, return_z = True):
        slope = (bbox.zmax - bbox.zmin)/(bbox.xmax - bbox.xmin)
        new_intercept = bbox.zmax - outer_thickness * 2    
        
        if return_z:
            return slope * inp + new_intercept
        else:
            return (inp - new_intercept)/slope
            
    
    inner = (
        cq.Workplane("XZ")
        .polyline([(-outer_thickness, outer_thickness), 
                   (-outer_thickness,outline_line(-outer_thickness)), 
                   (outline_line(outer_thickness, return_z = False), outer_thickness)])
        .close()
        .extrude(-5)
    )
    
    hollow_triangle = part.cut(inner)
    inner_area = max([f.Area() for f in inner.faces().vals()])
    
    return hollow_triangle, inner_area



In [26]:
def get_finray_infill(part, outline, spacing = 3, rod_diameter = 0.45):

    width = 70
    depth = 70
    thickness = 20

    grid = cq.Workplane("XZ")

    x = -width/2
    while x <= width/2:
        grid = grid.union(
            cq.Workplane("XZ").center(x, 25)
            .rect(rod_diameter, depth) 
            .extrude(-thickness)         
        )
        x += spacing

    grid = grid.rotate((0, 0, 0), (0, 1, 0), -45)
    
    infill_inside = grid.intersect(part)
    
    result = outline.union(infill_inside)
    
    return result
    
    
    

In [27]:
import math

def get_triangle_infill(part, outline, spacing = 5, rod_diameter = 0.45):
    
    width = spacing * 12  
    depth = spacing * 12 
    thickness = 20  
    
    grid = cq.Workplane("XZ")
    
    x = -width/2
    while x <= width/2:
        grid = grid.union(
            cq.Workplane("XZ").center(x, 3)
            .rect(rod_diameter, depth) 
            .extrude(-thickness)         
        )
        x += spacing

    grid = grid.rotate((0, 0, 0), (0, 1, 0), -90)

    grid_tri_one = cq.Workplane("XZ")

    x = -width/2
    while x <= width/2:
        grid_tri_one = grid_tri_one.union(
            cq.Workplane("XZ").center(x, 3)
            .rect(rod_diameter, depth) 
            .extrude(-thickness)         
        )
        x += spacing
        
    grid_tri_one = grid_tri_one.rotate((0, 0, 0), (0, 1, 0), 30)

    grid_tri_two = cq.Workplane("XZ")

    x = -width/2
    while x <= width/2:
        grid_tri_two = grid_tri_two.union(
            cq.Workplane("XZ").center(x, 3)
            .rect(rod_diameter, depth) 
            .extrude(-thickness)         
        )
        x += spacing
        
    grid_tri_two = grid_tri_two.rotate((0, 0, 0), (0, 1, 0), -30)

    grid_tri = grid_tri_two.union(grid_tri_one)
    grid = grid.union(grid_tri)
    
    grid = grid.rotate((0, 0, 0), (0, 1, 0), -45)

    infill_inside = grid.intersect(part)

    result = outline.union(infill_inside)
    
    return result, get_density(infill_inside)

In [28]:

def hexagon_func(side_length, thickness, cut=True):
    pts = []
    for i in range(6):
        angle = math.radians(60 * i)
        pts.append((math.cos(angle) * side_length,
                    math.sin(angle) * side_length))
    
    hexagon = cq.Workplane("XZ").polyline(pts + [pts[0]]).close().extrude(-20)
    hexagon = hexagon.rotate((0,0,0), (0, 1, 0), 30)
    if cut:
        hexagon = hexagon.cut(hexagon_func(side_length-thickness, thickness, cut=False))
        
    return hexagon
    
def get_honeycomb_infill(part, outline, side_length = 3, wall_thickness = 0.45):


    width = 50
    layers = math.ceil(40/(side_length))
    thickness = 20
    grid = cq.Workplane("XZ")
    space = math.sin(math.pi/3) * side_length
    
    for layer in range(layers):
        
        x = 0
        spacing = (math.cos(math.pi / 6) * side_length- (wall_thickness / 2)) if layer % 2 == 1 else 0
        while x >= -width/2:
            grid = grid.union(hexagon_func(side_length, wall_thickness).
                translate((x + spacing, 0, 40 - layer * (1.5 * side_length - wall_thickness))))
            x -= space*2 - wall_thickness


    grid = grid.rotate((0, 0, 0), (0, 1, 0), -5)
    
    infill_inside = grid.intersect(part)
    
    result = outline.union(infill_inside)

    return result, get_density(infill_inside)


In [29]:
def get_grid_infill(part, outline, spacing = 3.0, rod_diameter = 0.45):
    
    width = 80  
    depth = 80   
    thickness = 20  
    
    grid = cq.Workplane("XZ") 
    
    x = 0
    while x >= -width/2:
        grid = grid.union(
            cq.Workplane("XZ").center(x, 25)
            .rect(rod_diameter, depth) 
            .extrude(-thickness)         
        )
        x -= spacing
    
    z = -depth/2
    while z <= depth/5:
        grid = grid.union(
            cq.Workplane("XZ").center(0, z+25)
            .rect(width, rod_diameter)  
            .extrude(-thickness)         
        )
        z += spacing
    
    grid = grid.rotate((0, 0, 0), (0, 1, 0), 45)
    infill_inside = grid.intersect(part)
    result = outline.union(infill_inside)

    return result, get_density(infill_inside)





In [30]:
def get_density(infill_inside):
    
    slice_2d = infill_inside.section()
    slice_area = sum(f.Area() for f in slice_2d.faces().vals())
    return slice_area

In [32]:

triangle_outline_thickness = 0.87
infill_thickness = 0.45

part = cq.importers.importStep("STEP_files/GripperForOpt_v2.step")
outline, area = create_triangle_outline(part, outer_thickness=triangle_outline_thickness)
infill, density = get_triangle_infill(part, outline, spacing = 4) #0.87, 0.45, tessalate, engineering notebook
print(density/area * 100)
show(infill)


37.70867183845321
c


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…